# Sentence-Ablation Density Pilot on Llama-3.1-8B

**Goal:** test whether long-range linguistic influence is **dense** (many sentences contribute weakly), **sparse** (few sentences dominate, retrieval-like), or **hybrid** (broad graded field with salient anchors).

**Method:** for each target,
- Take the 1024-token prior context. Segment into sentences. Keep only targets with ≥10 prior sentences.
- Compute baseline target perplexity with full ordered context.
- For each prior sentence `s_i`, compute target perplexity with that sentence ABLATED (excised entirely from context).
- `ablation_influence[i] = ppl_ablated_i - ppl_full`. Positive = sentence contributed.
- Also compute SHUFFLED-REPLACEMENT variant: replace `s_i` with its own tokens shuffled in place (preserves vocabulary + length, destroys order). This isolates the order-specific contribution of each sentence.

**Corpora (pilot):** `gutenberg_fiction_en`, `ted_transcripts_en`. Buckeye is skipped because spontaneous-speech transcripts have no sentence boundaries (the splitter we use here would fall back to whole-doc, defeating the analysis). A chunk-based variant for spontaneous speech is a follow-up.

**Pilot N:** 20 targets per corpus. Total compute on H100 fp16: ~5 min (~50 ablations × 20 targets × 2 corpora × 2 variants).

**Output:** `My Drive/LRTIA/Results/sentence_ablation/llama/<corpus_id>.json`. Each record:
```
{ corpus_id, document_id, target_id,
  p_full, n_sentences,
  sentence_token_lengths: [...],
  sentence_distances: [...],          # tokens from end-of-sentence to start-of-target (nearest=0)
  sentence_text_snippets: [...],      # first 60 chars per sentence (for heatmap)
  ablation_influence: [...],          # per-sentence, ppl delta (positive = removing hurt)
  shufrepl_influence: [...] }         # per-sentence, ppl delta (positive = order mattered)
```

In [ ]:
!pip install -q -U accelerate

import numpy as np
import json, math, os, gc, random, re, time
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/sentence_ablation/llama'
BASE.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'

MAX_CTX = 1024
TARGET_LEN = 30
MIN_PRIOR_SENTS = 10        # skip targets whose 1024-tok context has fewer than this many sentences
MIN_DOC_TOK = MAX_CTX + TARGET_LEN + 50
DOCS_PER_CELL = 20          # pilot N
RUN_SHUFREPL = True         # run the shuffled-replacement variant
SEED = 20260503

# Pilot corpora — both have well-formed sentence punctuation.
RUN_CORPORA = ['gutenberg_fiction_en', 'ted_transcripts_en']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'MAX_CTX = {MAX_CTX} tokens, MIN_PRIOR_SENTS = {MIN_PRIOR_SENTS}')
print(f'Pilot: {DOCS_PER_CELL} docs/corpus × {len(RUN_CORPORA)} corpora '
      f'× ablation{" + shufrepl" if RUN_SHUFREPL else ""}')

In [ ]:
# Load model — fp16 for H100.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'{MODEL_NAME} loaded (fp16)')

In [ ]:
# ---- ppl pipeline ----
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

# ---- multilingual sentence splitter (matches sent-shuffle notebook) ----
SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[\u3002\uff01\uff1f])')
def split_sentences(text):
    parts = SENT_SPLIT_RE.split(text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= 4]

def tokenize_sentences(sentences):
    out = []
    for s in sentences:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            out.append((s, ids))
    return out  # list of (text, token_ids)

# ---- target selection: snap to sentence boundary at fraction 0.5; build 1024-tok prior ----
def select_target_and_prior_sentences(text, target_frac=0.5):
    sents = split_sentences(text)
    if len(sents) < MIN_PRIOR_SENTS + 2:
        return None
    sent_pairs = tokenize_sentences(sents)
    if len(sent_pairs) < MIN_PRIOR_SENTS + 2:
        return None
    sent_lens = [len(ids) for (_, ids) in sent_pairs]
    cum = [0]
    for L in sent_lens:
        cum.append(cum[-1] + L)
    n_total = cum[-1]
    if n_total < MIN_DOC_TOK:
        return None

    # Find target sentence: first sentence boundary at-or-after fractional position.
    target_pos_tok = int(target_frac * n_total)
    target_sent_i = next(
        (i for i in range(1, len(cum)) if cum[i - 1] >= target_pos_tok), None)
    if target_sent_i is None:
        return None
    # Walk forward until enough prior context.
    while target_sent_i < len(sent_pairs) and cum[target_sent_i - 1] < MAX_CTX:
        target_sent_i += 1
    if target_sent_i >= len(sent_pairs):
        return None

    # Build target: TARGET_LEN tokens starting at target sentence.
    tail = []
    for (_, s_ids) in sent_pairs[target_sent_i:]:
        tail.extend(s_ids)
        if len(tail) >= TARGET_LEN:
            break
    if len(tail) < TARGET_LEN:
        return None
    target_toks = tail[:TARGET_LEN]

    # Take the LAST sentences before target whose total tokens fit in MAX_CTX.
    prior = []  # list of (sentence_text, sentence_ids), oldest first
    total = 0
    for i in range(target_sent_i - 1, -1, -1):
        sl = sent_lens[i]
        if total + sl > MAX_CTX:
            break
        prior.insert(0, sent_pairs[i])
        total += sl
    if len(prior) < MIN_PRIOR_SENTS:
        return None

    # Compute distance from end-of-sentence to start-of-target (nearest = 0).
    n = len(prior)
    cum_lens_from_end = [0] * n
    running = 0
    for k in range(n - 1, -1, -1):
        cum_lens_from_end[k] = running
        running += len(prior[k][1])

    return {
        'target_toks': target_toks,
        'prior': prior,                                # list of (text, ids)
        'sentence_distances': cum_lens_from_end,
        'sentence_token_lengths': [len(ids) for (_, ids) in prior],
        'sentence_text_snippets': [t[:60] for (t, _) in prior],
    }

print('Pipeline + sentence selector ready')

In [ ]:
# ---- ablation + shuffled-replacement per sentence ----
def compute_per_sentence_influences(target_toks, prior, run_shufrepl=True):
    n = len(prior)
    full_ctx = [t for (_, ids) in prior for t in ids]
    p_full, _ = ppl_nll(full_ctx, target_toks)

    abl = []
    for i in range(n):
        # Ablation: drop sentence i entirely.
        ctx = [t for j, (_, ids) in enumerate(prior) if j != i for t in ids]
        p_abl, _ = ppl_nll(ctx, target_toks)
        abl.append(p_abl - p_full)

    sr = []
    if run_shufrepl:
        rng = random.Random(SEED)
        for i in range(n):
            shuf = list(prior[i][1])
            r = random.Random(SEED + i + 1)
            r.shuffle(shuf)
            ctx = [t for j, (_, ids) in enumerate(prior)
                   for t in (shuf if j == i else ids)]
            p_sr, _ = ppl_nll(ctx, target_toks)
            sr.append(p_sr - p_full)

    return p_full, abl, sr

print('Ablation pipeline ready')

In [ ]:
# ---- document resolver (corpus_expansion-format only for the pilot) ----
tok_manifest = {}
with open(TOK_MANIFEST_PATH) as f:
    for line in f:
        d = json.loads(line)
        tok_manifest[d['document_id']] = d['file_path']
ce_corpora = {}
with open(TARGETS_PATH) as f:
    for line in f:
        t = json.loads(line)
        ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])

def fix_ce_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

def get_doc_texts(corpus_id, limit=None):
    seen = 0
    for doc_id in sorted(ce_corpora.get(corpus_id, set())):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue
        abs_fp = Path(fix_ce_path(fp))
        if not abs_fp.exists():
            abs_fp = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try:
            yield doc_id, abs_fp.read_text(encoding='utf-8', errors='replace').strip()
        except Exception:
            continue
        seen += 1
        if limit and seen >= limit:
            return

for c in RUN_CORPORA:
    n = sum(1 for _ in get_doc_texts(c))
    print(f'  {c}: {n} total docs available')

In [ ]:
# ---- main run loop ----
for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    print(f'\n{"="*60}\n{corpus_id} (target {DOCS_PER_CELL} docs)\n{"="*60}')
    t0 = time.time()
    results = []
    skipped = 0

    docs_iter = get_doc_texts(corpus_id)  # all available; we iterate until DOCS_PER_CELL succeed
    pbar = tqdm(total=DOCS_PER_CELL, desc=corpus_id)
    for doc_id, text in docs_iter:
        if len(results) >= DOCS_PER_CELL:
            break
        sel = select_target_and_prior_sentences(text)
        if sel is None:
            skipped += 1; continue
        p_full, abl, sr = compute_per_sentence_influences(
            sel['target_toks'], sel['prior'], run_shufrepl=RUN_SHUFREPL)
        results.append({
            'corpus_id': corpus_id,
            'document_id': doc_id,
            'target_id': f'{doc_id}__pos50',
            'p_full': p_full,
            'n_sentences': len(sel['prior']),
            'sentence_token_lengths': sel['sentence_token_lengths'],
            'sentence_distances': sel['sentence_distances'],
            'sentence_text_snippets': sel['sentence_text_snippets'],
            'ablation_influence': abl,
            'shufrepl_influence': sr,
        })
        pbar.update(1)
    pbar.close()

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} targets in {elapsed/60:.1f} min ({skipped} skipped)')

    if results:
        all_abl = [v for r in results for v in r['ablation_influence']]
        pos = [v for v in all_abl if v > 0]
        print(f'  total sentences: {len(all_abl)}')
        print(f'  pct positive (ablation hurts): {100*len(pos)/len(all_abl):.1f}%')
        if pos:
            pos_sorted = sorted(pos, reverse=True)
            tot = sum(pos)
            print(f'  median positive ablation influence: {sorted(pos)[len(pos)//2]:.4f}')
            print(f'  top-1 share: {pos_sorted[0]/tot*100:.1f}%   '
                  f'top-5 share: {sum(pos_sorted[:5])/tot*100:.1f}%   '
                  f'top-10 share: {sum(pos_sorted[:10])/tot*100:.1f}%')

print('\nDone.')

In [ ]:
# ---- analysis + plots ----
import matplotlib.pyplot as plt

def gini(values):
    """Gini coefficient on non-negative values (0 = perfect equality, 1 = max inequality)."""
    v = sorted([x for x in values if x >= 0])
    n = len(v)
    if n == 0 or sum(v) == 0:
        return float('nan')
    s = sum((i + 1) * x for i, x in enumerate(v))
    return (2 * s) / (n * sum(v)) - (n + 1) / n

def summarize_cell(cell):
    fp = BASE / f'{cell}.json'
    if not fp.exists():
        print(f'{cell}: no cache')
        return None
    with open(fp) as f:
        recs = json.load(f)
    if not recs:
        return None

    all_abl = [v for r in recs for v in r['ablation_influence']]
    all_dist = [d for r in recs for d in r['sentence_distances']]
    all_sr = [v for r in recs for v in r['shufrepl_influence']] if recs[0].get('shufrepl_influence') else []

    pos = [v for v in all_abl if v > 0]
    pos_sr = [v for v in all_sr if v > 0]

    s = {
        'cell': cell,
        'n_targets': len(recs),
        'n_sentences': len(all_abl),
        'pct_positive_abl': 100 * len(pos) / len(all_abl) if all_abl else 0,
        'mean_abl': sum(all_abl) / len(all_abl) if all_abl else 0,
        'median_abl': sorted(all_abl)[len(all_abl) // 2] if all_abl else 0,
        'mean_pos_abl': sum(pos) / len(pos) if pos else 0,
        'median_pos_abl': sorted(pos)[len(pos) // 2] if pos else 0,
        'gini_pos_abl': gini(pos),
    }
    if pos:
        ps = sorted(pos, reverse=True)
        tot = sum(ps)
        s['top1_share'] = ps[0] / tot * 100
        s['top5_share'] = sum(ps[:5]) / tot * 100
        s['top10_share'] = sum(ps[:10]) / tot * 100
    if pos_sr:
        s['pct_positive_sr'] = 100 * len(pos_sr) / len(all_sr)
        s['mean_pos_sr'] = sum(pos_sr) / len(pos_sr)
    return s, recs, all_abl, all_dist, all_sr

summaries = []
for cell in RUN_CORPORA:
    out = summarize_cell(cell)
    if out:
        summaries.append(out)

print(f'{"cell":<26} {"N tgt":>6} {"N sent":>7} {"%pos":>6} '
      f'{"mean_pos":>9} {"med_pos":>8} {"top1%":>7} {"top5%":>7} '
      f'{"top10%":>7} {"Gini":>6}')
print('-' * 100)
for s, *_ in summaries:
    print(f'{s["cell"]:<26} {s["n_targets"]:>6} {s["n_sentences"]:>7} '
          f'{s["pct_positive_abl"]:>5.1f}% {s.get("mean_pos_abl",0):>9.4f} '
          f'{s.get("median_pos_abl",0):>8.4f} '
          f'{s.get("top1_share",0):>6.1f}% {s.get("top5_share",0):>6.1f}% '
          f'{s.get("top10_share",0):>6.1f}% {s.get("gini_pos_abl",float("nan")):>6.2f}')

In [ ]:
# ---- 5 diagnostic plots, side by side per corpus ----
for cell in RUN_CORPORA:
    out = summarize_cell(cell)
    if out is None: continue
    s, recs, all_abl, all_dist, all_sr = out
    pos = [v for v in all_abl if v > 0]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{cell}  —  {s["n_targets"]} targets, {s["n_sentences"]} sentences  |  '
                 f'%pos={s["pct_positive_abl"]:.1f}, top-5 share={s.get("top5_share",0):.1f}%, Gini={s.get("gini_pos_abl",0):.2f}',
                 fontsize=12, fontweight='bold')

    # 1. Ranked sentence influence curve (each target overlaid faintly + mean)
    ax = axes[0, 0]
    max_n = max(r['n_sentences'] for r in recs)
    ranked_means = [[] for _ in range(max_n)]
    for r in recs:
        ranked = sorted(r['ablation_influence'], reverse=True)
        ax.plot(range(len(ranked)), ranked, color='gray', alpha=0.15, linewidth=0.7)
        for i, v in enumerate(ranked):
            ranked_means[i].append(v)
    means = [sum(rm)/len(rm) if rm else 0 for rm in ranked_means]
    ax.plot(range(len(means)), means, color='C0', linewidth=2.5, label='mean across targets')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Sentence rank (1 = most influential)')
    ax.set_ylabel('Ablation influence (ppl drop)')
    ax.set_title('Ranked influence curve')
    ax.legend()

    # 2. Histogram of all sentence influences
    ax = axes[0, 1]
    ax.hist(all_abl, bins=50, color='C0', edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Ablation influence (ppl drop)')
    ax.set_ylabel('Count')
    ax.set_title(f'Histogram (red=0)  median={s["median_abl"]:.3f}, mean={s["mean_abl"]:.3f}')

    # 3. Influence vs distance from target (binned)
    ax = axes[0, 2]
    pairs = list(zip(all_dist, all_abl))
    bins = [0, 32, 64, 128, 256, 512, 1024]
    bin_labels = [f'{bins[i]}-{bins[i+1]}' for i in range(len(bins)-1)]
    bin_means, bin_meds, bin_pcts, bin_ns = [], [], [], []
    for i in range(len(bins) - 1):
        vs = [v for d, v in pairs if bins[i] <= d < bins[i + 1]]
        if vs:
            bin_means.append(sum(vs) / len(vs))
            bin_meds.append(sorted(vs)[len(vs) // 2])
            bin_pcts.append(100 * sum(1 for v in vs if v > 0) / len(vs))
            bin_ns.append(len(vs))
        else:
            bin_means.append(0); bin_meds.append(0); bin_pcts.append(0); bin_ns.append(0)
    x = range(len(bin_labels))
    ax.bar(x, bin_means, color='C0', alpha=0.7, label='mean influence')
    ax.plot(x, bin_meds, 'o-', color='C3', label='median')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(bin_labels, rotation=30)
    ax.set_xlabel('Distance from target (tokens)')
    ax.set_ylabel('Ablation influence')
    ax.set_title('Influence vs distance')
    ax.legend()

    # 4. Cumulative top-k contribution
    ax = axes[1, 0]
    if pos:
        ps = sorted(pos, reverse=True)
        tot = sum(ps)
        cum = []
        running = 0
        for v in ps:
            running += v
            cum.append(running / tot * 100)
        ax.plot(range(1, len(cum) + 1), cum, color='C0', linewidth=2)
        ax.set_xscale('log')
        for k in (1, 5, 10, 20):
            if k <= len(cum):
                ax.axvline(k, color='gray', linestyle=':', alpha=0.5)
                ax.annotate(f'top-{k}: {cum[k-1]:.0f}%', (k, cum[k-1]),
                            textcoords='offset points', xytext=(5, -10), fontsize=9)
        ax.set_xlabel('Top-k positive sentences')
        ax.set_ylabel('% of total positive influence')
        ax.set_title('Cumulative contribution')
        ax.grid(True, alpha=0.3)

    # 5. Example passage heatmap (one representative target — median total influence)
    ax = axes[1, 1]
    target_totals = [(i, sum(v for v in r['ablation_influence'] if v > 0))
                     for i, r in enumerate(recs)]
    target_totals.sort(key=lambda x: x[1])
    median_idx = target_totals[len(target_totals) // 2][0]
    r = recs[median_idx]
    infs = r['ablation_influence']
    n = len(infs)
    vmax = max(abs(min(infs)), abs(max(infs)))
    colors = [('green' if v > 0 else 'red') for v in infs]
    alphas = [min(1.0, abs(v) / (vmax + 1e-9)) for v in infs]
    for i, (txt, v, a) in enumerate(zip(r['sentence_text_snippets'], infs, alphas)):
        ax.barh(n - 1 - i, v, color=('C2' if v > 0 else 'C3'), alpha=0.4 + 0.6 * a)
        ax.text(0, n - 1 - i, f'  {txt}',
                fontsize=7, va='center', ha='left' if v < 0 else 'left')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Ablation influence')
    ax.set_ylabel('Sentence (newest at top)')
    ax.set_title(f'Example target  doc={r["document_id"][:30]}')
    ax.set_yticks([])

    # 6. Ablation vs shufrepl (per-sentence scatter, if available)
    ax = axes[1, 2]
    if all_sr:
        ax.scatter(all_abl, all_sr, alpha=0.4, s=15, color='C0')
        lo = min(min(all_abl), min(all_sr))
        hi = max(max(all_abl), max(all_sr))
        ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='y=x')
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
        ax.set_xlabel('Ablation influence (sentence removed)')
        ax.set_ylabel('Shufrepl influence (tokens shuffled)')
        ax.set_title(f'Ablation vs shufrepl per sentence  '
                     f'%sr_pos={s.get("pct_positive_sr",0):.1f}')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'No shufrepl data', ha='center', va='center',
                transform=ax.transAxes)

    plt.tight_layout()
    fig_path = BASE / f'{cell}_density.png'
    plt.savefig(fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  saved {fig_path.name}')

In [ ]:
# ---- interpretation cheat sheet ----
print('Reading the headline numbers:\n')
for s, *_ in summaries:
    print(f'  {s["cell"]}:')
    print(f'    {s["pct_positive_abl"]:.1f}% of sentences positively influence the target')
    print(f'    Top-5 sentences hold {s.get("top5_share",0):.1f}% of total positive influence')
    print(f'    Gini over positive influence: {s.get("gini_pos_abl",0):.2f}')
    if s.get('top5_share', 0) >= 80:
        verdict = 'SPARSE — a few hotspots dominate (retrieval-like signature)'
    elif s.get('top5_share', 0) <= 30 and s['pct_positive_abl'] >= 60:
        verdict = 'DENSE — many sentences contribute weakly (distributed influence)'
    else:
        verdict = 'HYBRID — broad positive influence with salient anchors'
    print(f'    Verdict: {verdict}\n')